<a href="https://colab.research.google.com/github/saveitkh/BarameyDabber/blob/main/voxcpm_server/VoxCPM2_Colab_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ VoxCPM2 Khmer — Server សម្រាប់ Khmer Dubber (Colab GPU)
**មុនចាប់ផ្ដើម:** Runtime ➔ Change runtime type ➔ **GPU** (T4 / L4 / A100 តាម Colab Pro) ➔ Save។
បន្ទាប់មកចុច **Runtime ➔ Run all** (ឬ Cell 1 ➔ 2)។

Server នេះគឺកូដតែមួយនឹង VPS (`voxcpm_server/app.py`) ដូច្នេះ Studio ប្រើបានគ្រប់មុខងារ៖ ធនាគារសំឡេង, ស្ដាប់ជាមុន, សំឡេងតួមិនជាន់គ្នា, ក្លូនពីវីដេអូ។

⚠️ **កូដសម្ងាត់ (`API_KEY`)**៖ ជ្រើសអក្សរវែងៗ ហើយដាក់ **ដូចគ្នា** ក្នុង `VOXCPM_API_KEY` លើ VPS។ បើទុកទទេ អ្នកណាដែលមាន Link ក៏ប្រើ GPU របស់អ្នកបាន។

In [2]:
# ===== CELL 1: ពិនិត្យ GPU + ដំឡើង =====
import subprocess, sys
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "❌ គ្មាន GPU — ប្ដូរ Runtime ទៅ GPU")
!apt-get -qq install -y ffmpeg libsndfile1 > /dev/null 2>&1
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
BRANCH = "claude/admiring-curie-90hbte"     # ប្ដូរទៅ "main" បន្ទាប់ពី merge
!rm -rf /content/BarameyDabber && git clone -q -b $BRANCH https://github.com/saveitkh/BarameyDabber.git /content/BarameyDabber
!pip install -q -r /content/BarameyDabber/voxcpm_server/requirements.txt
!pip install -q webrtcvad-wheels librosa && pip install -q --no-deps resemblyzer   # ឆ្មាំត្រួតពិនិត្យស្នាមសំឡេង
print("✅ ដំឡើងរួចរាល់")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 137.6 MB/s eta 0:00:00
✅ ដំឡើងរួចរាល់
GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-75349ac3-b9f3-49e5-a244-8a2aecfee237)

✅ ដំឡើងរួចរាល់


In [ ]:
# ===== CELL 2: បើក Server + Public URL (Cell នេះរត់ជាប់រហូត — កុំបិទ) =====
import os, re, sys, time, subprocess, requests

API_KEY      = "pangsokheng123456789"        # 👈 ដាក់កូដសម្ងាត់វែងៗ (ដូចគ្នានឹង VOXCPM_API_KEY លើ VPS)
USE_DRIVE    = True      # True = រក្សាសំឡេងដែលអ្នកបន្ថែមក្នុង Google Drive (មិនបាត់ពេល Colab បិទ)
OPTIMIZE     = True      # torch.compile — លឿនជាង តែចាប់ផ្ដើមយឺតជាងបន្តិច
TIMESTEPS    = 10        # GPU: អាចដាក់ 15-20 ឲ្យសំឡេងច្បាស់ជាង

voices = "/content/voices"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    voices = "/content/drive/MyDrive/KhmerDubber_voices"
os.makedirs(voices, exist_ok=True)

env = dict(os.environ, VOICE_DIR=voices, WORK_DIR="/content/work",
           BANK_SEED_DIR="/content/BarameyDabber/voxcpm_server/voice_bank",
           VOXCPM_API_KEY=API_KEY, VOXCPM_OPTIMIZE="1" if OPTIMIZE else "0",
           VOXCPM_DENOISER="0", SPEAKER_CHECK="1", VOXCPM_TRIES="3", VOXCPM_TIMESTEPS=str(TIMESTEPS))
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
                          cwd="/content/BarameyDabber/voxcpm_server", env=env,
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)

print("⏳ កំពុងផ្ទុកម៉ូដែល VoxCPM2 (លើកដំបូង ៥–១០ នាទី)...")
for _ in range(120):
    time.sleep(5)
    try:
        r = requests.get("http://127.0.0.1:8000/", timeout=3)
        if r.status_code == 200:
            print("✅ VoxCPM2 រួចរាល់ | សំឡេង:", list(r.json()["voices"])); break
    except Exception:
        pass
else:
    print("❌ ម៉ូដែលមិនឡើង — មើល:", subprocess.run("tail -20 /content/server.log", shell=True, capture_output=True, text=True).stdout)

log = "/content/cloudflared.log"
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=open(log, "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(0.5)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open(log, errors="ignore").read())
    if m: url = m.group(0); break

print("\n" + "=" * 64)
print("🌐 PUBLIC URL:", url or "⚠️ មិនទាន់ឃើញ — មើល /content/cloudflared.log")
print("👉 លើ VPS ដាក់ក្នុង /root/Project1/.env :")
print(f"   VOXCPM_URL={url}")
print(f"   VOXCPM_API_KEY={API_KEY or '(ទទេ)'}")
print("   បន្ទាប់មក: docker compose up -d studio")
print("=" * 64)
try:
    while server.poll() is None:
        time.sleep(30)
    print("🛑 Server ឈប់ — មើល /content/server.log")
except KeyboardInterrupt:
    server.terminate(); print("🛑 Stopped")